# Pandas Mastery: Data Wrangling for Machine Learning

Pandas is the backbone of data preparation in the Python data science ecosystem. It provides high-performance, easy-to-use data structures and tools for working with structured (tabular) data. Before any machine learning model can be trained, raw data must be cleaned, transformed, and reshaped — this is where Pandas excels.

In a typical ML workflow, 60–80% of the time is spent on data wrangling: handling missing values, converting data types, merging datasets, creating features, and filtering rows. Pandas makes these operations intuitive and vectorized, meaning you can express complex transformations in a single line of code without writing explicit loops.

This notebook covers the full spectrum of Pandas operations you will use daily as a data scientist: from loading data into Series and DataFrames, through advanced groupby operations and window functions, to performance optimization for large-scale data.

In [ ]:
import sys
from pathlib import Path
# Allow imports from the project root
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from scripts.data_loader import get_dataset, generate_synthetic_regression

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Series and DataFrame Creation

A **Series** is a one-dimensional labeled array that can hold any data type. A **DataFrame** is a two-dimensional labeled data structure with columns that can be of different types. Pandas provides numerous constructors and I/O functions to create these objects.

### From Python dicts and lists

In [ ]:
# Series from a list
s = pd.Series([10, 20, 30, 40, 50], name="values", index=["a", "b", "c", "d", "e"])
print("Series from list:")
print(s)

# DataFrame from a dict of lists
df_dict = pd.DataFrame({
    "product": ["Widget A", "Widget B", "Widget C", "Widget D"],
    "price": [9.99, 14.99, 24.99, 4.99],
    "stock": [150, 230, 75, 410],
    "category": ["electronics", "electronics", "home", "home"]
})
print("\nDataFrame from dict:")
print(df_dict)

In [ ]:
# DataFrame from a list of dicts (each row is a dict)
records = [
    {"city": "New York", "temp": 72, "humidity": 0.55},
    {"city": "Chicago", "temp": 68, "humidity": 0.60},
    {"city": "Los Angeles", "temp": 80, "humidity": 0.45},
]
df_records = pd.DataFrame(records)
print("DataFrame from list of dicts (records):")
print(df_records)

In [ ]:
# From NumPy arrays
arr = np.random.randn(5, 4)
df_np = pd.DataFrame(arr, columns=["A", "B", "C", "D"],
                     index=pd.date_range("2025-01-01", periods=5))
print("DataFrame from NumPy array:")
print(df_np)

In [ ]:
# Load from sklearn datasets via the project's data_loader
iris = get_dataset("iris")
print("\nIris dataset (first 5 rows):")
print(iris.head())

In [ ]:
# CSV: reading from a URL (Titanic)
try:
    titanic = pd.read_csv(
        "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    )
    print("Titanic loaded from CSV:", titanic.shape)
except Exception as e:
    print(f"Could not load Titanic via URL: {e}")
    # Fallback: synthetic data
    titanic = pd.DataFrame({
        "PassengerId": range(1, 101),
        "Survived": np.random.binomial(1, 0.4, 100),
        "Pclass": np.random.choice([1, 2, 3], 100),
        "Name": [f"Passenger {i}" for i in range(1, 101)],
        "Sex": np.random.choice(["male", "female"], 100),
        "Age": np.round(np.random.normal(30, 12, 100), 1),
        "Fare": np.round(np.random.exponential(50, 100), 2),
        "Embarked": np.random.choice(["S", "C", "Q"], 100)
    })
    print("Using synthetic Titanic-like data:", titanic.shape)

print(titanic.head())

In [ ]:
# Excel and JSON
import io
# Write a small Excel file in memory to demonstrate
buffer = io.BytesIO()
df_dict.to_excel(buffer, index=False, sheet_name="Products")
buffer.seek(0)
df_excel = pd.read_excel(buffer, sheet_name="Products")
print("Read back from Excel:")
print(df_excel)

# JSON
json_str = df_dict.to_json(orient="records")
df_json = pd.read_json(io.StringIO(json_str))
print("\nRead back from JSON:")
print(df_json)

## 2. Inspecting Data

Before doing any manipulation, you need to understand the structure, content, and quality of your data. Pandas provides a rich set of inspection methods.

In [ ]:
df = get_dataset("wine")

print("First 5 rows:")
display(df.head())

print("\nDataFrame info:")
df.info()

print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

In [ ]:
# Descriptive statistics
print("Summary statistics:")
display(df.describe())

print("\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique")

print("\nValue counts for target:")
print(df["target"].value_counts())

print("\nMemory usage (MB):")
print(df.memory_usage(deep=True) / 1024**2)

## 3. Indexing: Selecting and Filtering Data

Pandas offers multiple indexing paradigms. Master these to express any row/column selection concisely.

In [ ]:
df = get_dataset("housing")
print("Columns:", df.columns.tolist())

# .loc: label-based indexing
print("\n.loc[0:5, ['MedInc', 'HouseAge']]:")
print(df.loc[0:5, ["MedInc", "HouseAge"]])

# .iloc: integer-position-based indexing
print("\n.iloc[:5, :3]:")
print(df.iloc[:5, :3])

# Boolean indexing
print("\nHouses with > 5 rooms and median income > 5:")
subset = df[(df["AveRooms"] > 5) & (df["MedInc"] > 5)]
print(f"Rows: {len(subset)}")
print(subset.head())

In [ ]:
# .query() for expressive filtering
query_result = df.query("AveRooms > 5 and MedInc > 5 and HouseAge < 30")
print(f".query() returned {len(query_result)} rows")
print(query_result.head())

# .filter() for column selection by name pattern
filtered = df.filter(regex="^(Med|House)")
print("\n.filter(regex) columns:", filtered.columns.tolist())
print(filtered.head())

## 4. Handling Missing Data

Real-world datasets almost always have missing values. Pandas represents these as `NaN` (for numeric) or `None` (for Python objects). The key decisions are: which rows/columns to drop, and what values to fill in their place.

In [ ]:
# Create a dataset with intentional missing values
np.random.seed(42)
n = 100
df_miss = pd.DataFrame({
    "id": range(n),
    "age": np.random.randint(18, 80, n).astype(float),
    "income": np.round(np.random.normal(60000, 15000, n), 0),
    "education_years": np.random.randint(8, 22, n).astype(float),
    "city": np.random.choice(["NYC", "LA", "CHI", "HOU"], n),
})
# Introduce missing values
miss_idx = np.random.choice(n, 15, replace=False)
df_miss.loc[miss_idx[:7], "age"] = np.nan
df_miss.loc[miss_idx[7:12], "income"] = np.nan
df_miss.loc[miss_idx[12:], "education_years"] = np.nan

print("Missing value counts:\n")
print(df_miss.isna().sum())
print(f"\nTotal missing: {df_miss.isna().sum().sum()}")

In [ ]:
# Drop strategies
print("Original shape:", df_miss.shape)
print("Drop any row with a NaN:", df_miss.dropna().shape)
print("Drop rows where all values are NaN:", df_miss.dropna(how="all").shape)
print("Drop rows with < 3 non-NA:", df_miss.dropna(thresh=3).shape)
print("Drop columns with any NaN:", df_miss.dropna(axis=1).shape)

In [ ]:
# Fill strategies
df_filled = df_miss.copy()

# Fill age with mean
df_filled["age"] = df_filled["age"].fillna(df_filled["age"].mean())
print(f"Age filled with mean: {df_filled['age'].isna().sum()} missing remaining")

# Fill income with median (more robust to outliers)
df_filled["income"] = df_filled["income"].fillna(df_filled["income"].median())
print(f"Income filled with median: {df_filled['income'].isna().sum()} missing remaining")

# Fill education_years with mode (most common value)
mode_val = df_filled["education_years"].mode()[0]
df_filled["education_years"] = df_filled["education_years"].fillna(mode_val)
print(f"Education filled with mode ({mode_val}): {df_filled['education_years'].isna().sum()} missing remaining")

print("\nAll missing values have been filled:")
print(df_filled.isna().sum())

In [ ]:
# Forward fill and backward fill
ts_data = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=10, freq="D"),
    "value": [100, np.nan, np.nan, 105, np.nan, 110, np.nan, np.nan, np.nan, 120]
})

print("Original:")
print(ts_data)

ts_data["ffill"] = ts_data["value"].ffill()
ts_data["bfill"] = ts_data["value"].bfill()
ts_data["interpolated"] = ts_data["value"].interpolate()

print("\nWith forward fill, backward fill, and interpolation:")
print(ts_data)

## 5. Data Types and Conversion

Correct data types are critical for memory efficiency and proper computation. Pandas often infers types on import, but you will need to cast columns to the right type for your analysis.

In [ ]:
df = get_dataset("cancer")
print("Original dtypes:")
print(df.dtypes.value_counts())

# Convert target to categorical
df["target"] = df["target"].astype("category")
print("\nAfter converting target to category:")
print(df["target"].dtype)
print(df["target"].cat.categories.tolist())

In [ ]:
# to_numeric: safely convert strings that should be numbers
messy = pd.Series(["1", "2.5", "3.0", "four", "5", ""])
cleaned = pd.to_numeric(messy, errors="coerce")
print("Original:", messy.tolist())
print("Numeric (coerce):", cleaned.tolist())

# to_datetime
dates = pd.Series(["2025-01-15", "2025/02/20", "20250310", "invalid", "15-Jan-2025"])
parsed = pd.to_datetime(dates, errors="coerce")
print("\nParsed dates:")
print(parsed)

# Downcast numeric types to save memory
df_num = df.select_dtypes(include=[np.number])
print("\nMemory before downcasting:", df_num.memory_usage(deep=True).sum() / 1024**2, "MB")
for col in df_num.columns:
    df[col] = pd.to_numeric(df[col], downcast="float" if df[col].dtype == "float64" else "integer")
print("Memory after downcasting:", df.memory_usage(deep=True).sum() / 1024**2, "MB")

## 6. Adding and Removing Columns

Feature engineering often involves creating new columns and dropping irrelevant ones.

In [ ]:
df = get_dataset("housing").iloc[:100].copy()

# Using assign (returns a new DataFrame, doesn't modify in place)
df2 = df.assign(
    rooms_per_person=df["AveRooms"] / df["AveOccup"],
    income_bracket=pd.cut(df["MedInc"], bins=[0, 3, 6, 20], labels=["Low", "Mid", "High"]),
    house_age_group=np.where(df["HouseAge"] > 30, "Old", "New")
)
print("New columns via assign:")
print(df2[["AveRooms", "AveOccup", "rooms_per_person", "income_bracket", "house_age_group"]].head())

In [ ]:
# Insert at a specific position
df.insert(1, "MedInc_log", np.log1p(df["MedInc"]))
print("After insert at position 1:")
print(df.columns.tolist())
print(df[["MedInc", "MedInc_log"]].head())

# Drop columns
df_dropped = df.drop(columns=["MedInc_log"])
print("\nAfter dropping MedInc_log:", df_dropped.columns.tolist())

# Drop rows by index
df_no_first = df.drop(index=[0, 1, 2])
print(f"After dropping first 3 rows: {df_no_first.shape[0]} rows")

## 7. Filtering and Sorting

Beyond basic boolean indexing, Pandas offers specialized filtering methods that are both readable and efficient.

In [ ]:
df = get_dataset("housing").iloc[:200].copy()

# sort_values
sorted_df = df.sort_values("MedInc", ascending=False)
print("Top 5 by MedInc:")
print(sorted_df[["MedInc", "HouseAge", "AveRooms"]].head())

# nlargest / nsmallest
print("\n5 most expensive houses:")
print(df["MedHouseVal"].nlargest(5))

print("\n5 cheapest houses:")
print(df["MedHouseVal"].nsmallest(5))

In [ ]:
# between
mid_income = df[df["MedInc"].between(3, 6)]
print(f"Rows with MedInc between 3 and 6: {len(mid_income)}")

# isin
df["ocean_proximity"] = np.random.choice(["NEAR BAY", "INLAND", "NEAR OCEAN", "ISLAND"], len(df))
coastal = df[df["ocean_proximity"].isin(["NEAR BAY", "NEAR OCEAN"])]
print(f"Coastal properties: {len(coastal)}")

# str.contains
bay_related = df[df["ocean_proximity"].str.contains("BAY", case=False)]
print(f"Properties with 'BAY' in proximity: {len(bay_related)}")

## 8. GroupBy: Split-Apply-Combine

The groupby operation is a three-step process: split the data into groups, apply a function to each group independently, and combine the results into a single data structure. This is the foundation of aggregate analysis.

In [ ]:
np.random.seed(42)
df_group = pd.DataFrame({
    "department": np.random.choice(["Sales", "Engineering", "HR", "Marketing"], 200),
    "team": np.random.choice(["Alpha", "Beta", "Gamma"], 200),
    "salary": np.round(np.random.normal(75000, 20000, 200), 0),
    "experience": np.random.randint(1, 20, 200),
    "projects": np.random.randint(0, 8, 200),
})

print("Basic groupby - mean salary by department:")
print(df_group.groupby("department")["salary"].mean().round(0))

In [ ]:
# Multiple aggregations at once
agg_result = df_group.groupby("department").agg({
    "salary": ["mean", "std", "min", "max"],
    "experience": "mean",
    "projects": "sum"
}).round(1)
print("Multiple aggregations:")
print(agg_result)

In [ ]:
# Groupby with multiple keys, named aggregation (Pandas 0.25+)
named_agg = df_group.groupby(["department", "team"], as_index=False).agg(
    avg_salary=("salary", "mean"),
    max_projects=("projects", "max"),
    headcount=("salary", "count"),
    total_experience=("experience", "sum")
)
print("Named aggregation (department + team):")
print(named_agg.round(1).head(10))

In [ ]:
# transform: broadcast group-level values back to the original rows
df_group["dept_avg_salary"] = df_group.groupby("department")["salary"].transform("mean")
df_group["salary_vs_dept_avg"] = df_group["salary"] - df_group["dept_avg_salary"]
print("Transform example (salary relative to department average):")
print(df_group[["department", "salary", "dept_avg_salary", "salary_vs_dept_avg"]].head(10))

In [ ]:
# filter: keep groups that satisfy a condition
filtered_groups = df_group.groupby("department").filter(
    lambda x: x["salary"].mean() > 75000
)
print(f"Original: {len(df_group)} rows")
print(f"After filtering out departments with avg salary <= 75k: {len(filtered_groups)} rows")
remaining_depts = filtered_groups["department"].unique()
print(f"Remaining departments: {remaining_depts}")

## 9. Merging and Joining

Real data lives in multiple tables. Merging combines DataFrames by key columns (like SQL JOINs), while concatenation stacks them along rows or columns.

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "city": ["NYC", "LA", "NYC", "Chicago", "LA"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 2, 4, 6],
    "amount": [250, 175, 300, 400, 150],
    "order_date": pd.date_range("2025-01-01", periods=5, freq="W")
})

print("Customers:")
print(customers)
print("\nOrders:")
print(orders)

In [ ]:
# Inner join: only matching keys
inner = pd.merge(customers, orders, on="customer_id", how="inner")
print("Inner join:")
print(inner)

# Left join: all customers, even those without orders
left = pd.merge(customers, orders, on="customer_id", how="left")
print("\nLeft join:")
print(left)

# Right join: all orders, even customers missing
right = pd.merge(customers, orders, on="customer_id", how="right")
print("\nRight join:")
print(right)

# Outer join: everything
outer = pd.merge(customers, orders, on="customer_id", how="outer")
print("\nOuter join:")
print(outer)

In [ ]:
# Merging on different column names
orders_renamed = orders.rename(columns={"customer_id": "cid"})
merged_diff = pd.merge(customers, orders_renamed, left_on="customer_id", right_on="cid", how="left")
print("Merge with different key names:")
print(merged_diff)

# Concat: stacking DataFrames
q1 = pd.DataFrame({"month": ["Jan", "Feb", "Mar"], "sales": [100, 120, 140]})
q2 = pd.DataFrame({"month": ["Apr", "May", "Jun"], "sales": [160, 180, 200]})
year_half = pd.concat([q1, q2], ignore_index=True)
print("\nConcatenated vertically:")
print(year_half)

# Concat horizontally
info = pd.DataFrame({"month": ["Jan", "Feb", "Mar"], "target": [110, 130, 150]})
combined = pd.concat([q1.set_index("month"), info.set_index("month")], axis=1)
print("\nConcatenated horizontally:")
print(combined)

## 10. Pivot Tables and Cross-Tabulations

Pivot tables reshape data by grouping and aggregating along two dimensions. They are essential for creating summary matrices and transforming long data into wide format.

In [ ]:
np.random.seed(42)
sales = pd.DataFrame({
    "region": np.random.choice(["North", "South", "East", "West"], 500),
    "product": np.random.choice(["A", "B", "C"], 500),
    "quarter": np.random.choice(["Q1", "Q2", "Q3", "Q4"], 500),
    "revenue": np.round(np.random.uniform(1000, 10000, 500), 2),
    "units": np.random.randint(10, 500, 500),
})

# pivot_table: aggregate with multiple values and functions
pt = pd.pivot_table(
    sales,
    values=["revenue", "units"],
    index="region",
    columns="quarter",
    aggfunc={"revenue": "mean", "units": "sum"},
    margins=True,
    margins_name="Total"
)
print("Pivot table - avg revenue and total units by region and quarter:")
print(pt.round(1))

In [ ]:
# crosstab: frequency tables
ct = pd.crosstab(
    index=sales["region"],
    columns=sales["product"],
    values=sales["units"],
    aggfunc="sum",
    normalize="index",
    margins=True
).round(3)
print("Cross-tabulation (unit distribution normalized by region):")
print(ct)

# Simple frequency table
freq = pd.crosstab(sales["region"], sales["quarter"])
print("\nFrequency of region-quarter combinations:")
print(freq)

In [ ]:
# melt: wide to long format
wide = pd.DataFrame({
    "id": [1, 2, 3],
    "test_a": [85, 90, 78],
    "test_b": [88, 92, 80],
    "test_c": [82, 95, 74]
})
print("Wide format:")
print(wide)

long = wide.melt(id_vars=["id"], var_name="test", value_name="score")
print("\nLong format via melt:")
print(long)

In [ ]:
# stack / unstack: pivot along the index
df_stack = sales.groupby(["region", "product"])["revenue"].mean().round(0)
print("MultiIndex Series (grouped):")
print(df_stack)

unstacked = df_stack.unstack(level="product")
print("\nUnstacked (product went to columns):")
print(unstacked)

stacked_back = unstacked.stack()
print("\nStacked back:")
print(stacked_back)

## 11. Window Operations: Rolling, Expanding, and Apply

Window functions operate over a sliding segment of your data. They are indispensable for time-series feature engineering: rolling means, expanding sums, and custom windowed computations.

In [ ]:
# Create a time series
np.random.seed(42)
ts = pd.DataFrame(
    {"value": 100 + np.cumsum(np.random.randn(60) * 2)},
    index=pd.date_range("2025-01-01", periods=60, freq="D")
)

print("Time series (first 10):")
print(ts.head(10))

# Rolling mean (7-day window)
ts["rolling_mean_7"] = ts["value"].rolling(window=7).mean()
ts["rolling_std_7"] = ts["value"].rolling(window=7).std()

print("\nWith 7-day rolling mean and std:")
print(ts.head(14))

In [ ]:
# Expanding window: cumulative statistics
ts["expanding_mean"] = ts["value"].expanding().mean()
ts["expanding_max"] = ts["value"].expanding().max()

# Rolling apply with a custom function
def range_pct(series):
    """Return the range as a percentage of the mean."""
    return (series.max() - series.min()) / series.mean() * 100

ts["rolling_range_pct"] = ts["value"].rolling(window=14).apply(range_pct, raw=False)

print("Expanding and rolling apply (select columns):")
print(ts.iloc[20:30, [0, 3, 4, 5]].round(2))

In [ ]:
# Rolling on grouped data
multi_ts = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=60, freq="D").repeat(2),
    "group": ["A", "B"] * 60,
    "value": np.random.randn(120).cumsum() + 50
})

multi_ts["rolling_mean_by_group"] = (
    multi_ts.groupby("group")["value"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
print("Rolling mean computed per group:")
print(multi_ts.head(12))

## 12. String Operations with .str Accessor

Pandas exposes vectorized string methods through the `.str` accessor, letting you process text columns without writing slow loops.

In [ ]:
text_data = pd.Series([
    "Machine Learning is transforming industries.",
    "  Data Science requires clean data.  ",
    "NLP: Natural Language Processing",
    "Python for data analysis",
    "DEEP LEARNING with TensorFlow",
    ""
])

print("Original:", text_data.tolist(), "\n")

print("Lowercase:")
print(text_data.str.lower().tolist())

print("\nStrip whitespace:")
print(text_data.str.strip().tolist())

print("\nContains 'Data' (case insensitive):")
print(text_data.str.contains("data", case=False).tolist())

print("\nExtract first word:")
print(text_data.str.extract(r"(\w+)").tolist())

print("\nReplace spaces with underscores:")
print(text_data.str.replace(" ", "_", regex=False).tolist())

print("\nSplit into words:")
print(text_data.str.split().tolist())

print("\nCharacter length:")
print(text_data.str.len().tolist())

In [ ]:
# Practical example: cleaning a dirty column
dirty = pd.DataFrame({
    "email": [
        "  Alice@Example.com  ",
        "BOB@example.com",
        "Charlie@EXAMPLE.ORG",
        None,
        "diana@example.net",
    ],
    "phone": [
        "(212) 555-1234",
        "212.555.5678",
        "212-555-9012",
        "5551234",
        "N/A",
    ]
})

# Normalize emails
dirty["email_clean"] = dirty["email"].str.strip().str.lower()

# Extract just digits from phone
dirty["phone_digits"] = dirty["phone"].str.replace(r"\D", "", regex=True)
dirty["phone_digits"] = dirty["phone_digits"].replace({"": None, "N": None})

# Extract domain from email
dirty["email_domain"] = dirty["email_clean"].str.extract(r"@(.+)")

print("Cleaned data:")
print(dirty)

## 13. Working with Datetime

DateTime handling is essential for time-based analysis. The `.dt` accessor provides component extraction, and `resample` enables time-based aggregations.

In [ ]:
np.random.seed(42)
n_points = 1000
dates = pd.date_range("2024-01-01", periods=n_points, freq="h")
ts_full = pd.DataFrame({
    "timestamp": dates,
    "value": np.random.randn(n_points).cumsum() + 100,
    "category": np.random.choice(["A", "B", "C"], n_points)
})

# Extract datetime components via .dt accessor
ts_full["year"] = ts_full["timestamp"].dt.year
ts_full["month"] = ts_full["timestamp"].dt.month
ts_full["day"] = ts_full["timestamp"].dt.day
ts_full["hour"] = ts_full["timestamp"].dt.hour
ts_full["weekday"] = ts_full["timestamp"].dt.weekday
ts_full["is_weekend"] = ts_full["weekday"].isin([5, 6]).astype(int)
ts_full["quarter"] = ts_full["timestamp"].dt.quarter

print("Datetime components (first 5 rows):")
print(ts_full.head())

In [ ]:
# Resample: aggregate by time period
ts_series = ts_full.set_index("timestamp")["value"]

daily = ts_series.resample("D").agg(["mean", "min", "max", "std"]).round(2)
print("Daily resample:")
print(daily.head())

# Weekly with multiple aggregations
weekly = ts_series.resample("W").agg(["mean", "sum", "count"]).round(2)
print("\nWeekly resample:")
print(weekly.head())

In [ ]:
# GroupBy on time periods
ts_full["week_start"] = ts_full["timestamp"].dt.to_period("W").dt.start_time

weekly_by_category = (
    ts_full.groupby([pd.Grouper(key="timestamp", freq="W"), "category"])["value"]
    .agg(["mean", "count"])
    .round(2)
)
print("Weekly aggregations by category:")
print(weekly_by_category.head(12))

## 14. Applying Functions

When you need to transform data element-wise, row-wise, or column-wise, Pandas provides a family of `apply` methods. While vectorized operations are always preferred, `apply` is indispensable for complex logic that cannot be expressed as a NumPy ufunc.

In [ ]:
df = get_dataset("housing").iloc[:50].copy()

# apply on a column (element-wise, but slower than vectorized)
df["MedInc_category"] = df["MedInc"].apply(
    lambda x: "Low" if x < 3 else ("Medium" if x < 6 else "High")
)
print("apply on a column:")
print(df[["MedInc", "MedInc_category"]].head(10))

In [ ]:
# apply on rows (axis=1)
def score_house(row):
    """Compute a custom quality score."""
    score = 0
    if row["MedInc"] > 5:
        score += 1
    if row["HouseAge"] < 20:
        score += 1
    if row["AveRooms"] > 5:
        score += 1
    if row["AveOccup"] < 3:
        score += 1
    return score

df["quality_score"] = df.apply(score_house, axis=1)
print("Row-wise apply (quality score):")
print(df[["MedInc", "HouseAge", "AveRooms", "AveOccup", "quality_score"]].head(10))

In [ ]:
# map: transform values in a Series using a dict
rating_map = {0: "Poor", 1: "Fair", 2: "Good", 3: "Very Good", 4: "Excellent"}
df["quality_label"] = df["quality_score"].map(rating_map)
print("map with a dict:")
print(df[["quality_score", "quality_label"]].drop_duplicates().sort_values("quality_score"))

# applymap: element-wise on a DataFrame (deprecated in newer versions, use .map() instead)
# Modern approach: use .map() on each column or vectorized operations
df_subset = df[["MedInc", "HouseAge", "AveRooms"]].head(5)
print("\nOriginal subset:")
print(df_subset)
print("\nRounded:")
print(df_subset.map(lambda x: round(x, 1)))

In [ ]:
# pipe: chain operations cleanly
def add_room_feature(df):
    return df.assign(rooms_per_person=df["AveRooms"] / df["AveOccup"])

def add_income_bracket(df):
    return df.assign(
        income_bracket=pd.cut(df["MedInc"], bins=[0, 3, 6, 20], labels=["Low", "Mid", "High"])
    )

def select_columns(df):
    return df[["MedInc", "AveRooms", "AveOccup", "rooms_per_person", "income_bracket"]]

pipeline = (
    df
    .pipe(add_room_feature)
    .pipe(add_income_bracket)
    .pipe(select_columns)
)
print("Piped transformations:")
print(pipeline.head())

## 15. Performance Tips

Pandas is fast, but careless use of `apply` and object dtypes can tank performance. Here are the most impactful optimizations.

In [ ]:
import time

n = 100_000
df_perf = pd.DataFrame({
    "x": np.random.randn(n),
    "y": np.random.randn(n),
    "category": np.random.choice(["low", "medium", "high"], n),
})

# Vectorized operation vs apply
start = time.time()
result_vec = df_perf["x"] * df_perf["y"] + np.sqrt(np.abs(df_perf["x"]))
time_vec = time.time() - start

start = time.time()
result_apply = df_perf.apply(lambda row: row["x"] * row["y"] + np.sqrt(abs(row["x"])), axis=1)
time_apply = time.time() - start

print(f"Vectorized: {time_vec:.4f}s")
print(f"apply (axis=1): {time_apply:.4f}s")
print(f"Speedup: {time_apply / time_vec:.1f}x")

In [ ]:
# Categorical dtype saves memory and speeds up groupby
start = time.time()
groupby_obj = df_perf.groupby("category")["x"].mean()
time_object = time.time() - start

df_perf["category_cat"] = df_perf["category"].astype("category")
start = time.time()
groupby_cat = df_perf.groupby("category_cat")["x"].mean()
time_cat = time.time() - start

print(f"GroupBy with object dtype: {time_object:.4f}s")
print(f"GroupBy with categorical: {time_cat:.4f}s")

# Memory comparison
obj_mem = df_perf["category"].memory_usage(deep=True) / 1024
cat_mem = df_perf["category_cat"].memory_usage(deep=True) / 1024
print(f"\nObject column memory: {obj_mem:.1f} KB")
print(f"Categorical column memory: {cat_mem:.1f} KB")
print(f"Memory reduction: {(1 - cat_mem/obj_mem) * 100:.1f}%")

In [ ]:
# Use inplace=False (returns a copy) for chaining
# Efficient column selection: pass a list, not chained .dot notation
print("Memory usage of DataFrame:")
print(df_perf.memory_usage(deep=True) / 1024, "KB")

# Downcast where safe
for col in ["x", "y"]:
    df_perf[col] = pd.to_numeric(df_perf[col], downcast="float")
print("\nAfter downcast:")
print(df_perf.memory_usage(deep=True) / 1024, "KB")

## 16. Working with Large Datasets

When datasets exceed available RAM, Pandas provides strategies to stay efficient: chunked reading, optimized dtypes, and selective column loading.

In [ ]:
# Simulate a large dataset by writing a CSV to a string buffer
import io

large_n = 500_000
large_df = pd.DataFrame({
    "id": range(large_n),
    "value_a": np.random.randn(large_n),
    "value_b": np.random.randn(large_n),
    "category": np.random.choice(["X", "Y", "Z"], large_n),
    "date": pd.date_range("2020-01-01", periods=large_n, freq="min")[:large_n]
})

buffer = io.StringIO()
large_df.to_csv(buffer, index=False)
buffer.seek(0)

print("Full dataset memory:")
print(f"{large_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Chunked reading
buffer.seek(0)
chunks = pd.read_csv(buffer, chunksize=50_000)

chunk_means = []
for i, chunk in enumerate(chunks):
    chunk_means.append(chunk["value_a"].mean())
    print(f"Processed chunk {i+1}: {len(chunk)} rows, mean value_a = {chunk['value_a'].mean():.4f}")

overall_mean = np.mean(chunk_means)
print(f"\nOverall mean from chunks: {overall_mean:.4f}")
print(f"Direct mean for validation: {large_df['value_a'].mean():.4f}")

In [ ]:
# Optimize dtypes at read time
buffer.seek(0)
dtype_spec = {
    "id": "int32",
    "value_a": "float32",
    "value_b": "float32",
    "category": "category",
    "date": "object"  # parse as string, convert later
}

df_opt = pd.read_csv(buffer, dtype=dtype_spec, parse_dates=["date"])
print("Optimized dtypes:")
print(df_opt.dtypes)

original_mem = large_df.memory_usage(deep=True).sum() / 1024**2
optimized_mem = df_opt.memory_usage(deep=True).sum() / 1024**2
print(f"\nOriginal memory: {original_mem:.2f} MB")
print(f"Optimized memory: {optimized_mem:.2f} MB")
print(f"Reduction: {(1 - optimized_mem / original_mem) * 100:.1f}%")

In [ ]:
# Read only specific columns to save memory
buffer.seek(0)
df_subset = pd.read_csv(buffer, usecols=["id", "value_a", "category"])
print(f"Subset columns: {df_subset.columns.tolist()}")
print(f"Subset memory: {df_subset.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Full memory: {original_mem:.2f} MB")
print(f"Memory with subset: {(1 - df_subset.memory_usage(deep=True).sum() / large_df.memory_usage(deep=True).sum()) * 100:.1f}% less")

## 17. Practice Exercises

Test your Pandas skills with these challenges. Try solving each one using the operations covered above.

### Exercise 1: Data Cleaning

Load a dataset of your choice. Identify columns with missing values and implement a strategy to handle each one. Justify why you chose each fill method (mean, median, mode, forward fill, or interpolation).

In [ ]:
# Your solution here
# import sys; sys.path.insert(0, str(Path.cwd().parent))
# from scripts.data_loader import get_dataset
# df = get_dataset("titanic")
# ... your data cleaning ...
print("Run your cleaning pipeline here")

### Exercise 2: Feature Engineering

Using the California Housing dataset, create at least 3 new features (e.g., room density, age groups, income-to-rooms ratio). Then group by the income bracket and compute mean values for each new feature.

In [ ]:
# Your solution here
print("Write your feature engineering pipeline here")

### Exercise 3: Merging and Aggregation

Create two DataFrames — `employees` (id, name, dept_id, salary) and `departments` (dept_id, dept_name, budget). Merge them, then find the department with the highest average salary and the one with the most employees.

In [ ]:
# Your solution here
print("Build and merge employee-department data here")

### Exercise 4: Time Series Resampling

Generate hourly data for 90 days. Resample it to 3-day rolling averages. Also compute the weekly sum and compare the trend against the raw hourly data. Use the .dt accessor to flag weekend vs weekday values.

In [ ]:
# Your solution here
print("Create hourly time series and resample here")

### Exercise 5: Pivot and Melt

Create a wide DataFrame with sales data for 3 products across 4 quarters. Use melt to convert it to long format. Then use pivot_table to reconstruct a summary with product as rows, quarter as columns, and mean sales as values. Add grand totals using the margins parameter.

In [ ]:
# Your solution here
print("Wide-to-long and pivot exercise here")